In [98]:
import numpy as np
import pandas as pd
from itertools import combinations
import random

In [99]:
df = pd.read_csv("./data/prova_cn.csv")

In [100]:
def levenshtein_distance(a, b):

    n, m = len(a), len(b)

    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i

    for j in range(m+1):
        dp[0][j] = j

    for i in range(1,n+1):
        for j in range(1,m+1):

            cost = 0 if a[i-1] == b[j-1] else 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[n][m]

In [101]:
def similarity(seq1, seq2):

    N = len(seq1)

    dist = levenshtein_distance(seq1, seq2)

    sim = ((N - dist) / N) * 100

    return sim

In [102]:
def kmeans_levenshtein(data, k=2, max_iter=10):

    sequences = data.iloc[:,1:].values.tolist()

    centers = random.sample(sequences, k)

    for _ in range(max_iter):

        clusters = {i:[] for i in range(k)}

        for seq in sequences:

            distances = [levenshtein_distance(seq,c) for c in centers]

            cluster_id = np.argmin(distances)

            clusters[cluster_id].append(seq)

        new_centers = []

        for cluster in clusters.values():

            if len(cluster)==0:
                new_centers.append(random.choice(sequences))
                continue

            center=[]

            for i in range(len(cluster[0])):

                values=[seq[i] for seq in cluster]

                center.append(max(set(values), key=values.count))

            new_centers.append(center)

        centers=new_centers

    return clusters

In [103]:
clusters = kmeans_levenshtein(df, k=2)

clusters

{0: [['2025-07-17 22:01:15',
   2376,
   '2a',
   2,
   3,
   4,
   3.0,
   4,
   1,
   3,
   2,
   2,
   3,
   1,
   5,
   5,
   3,
   1,
   4,
   5,
   1,
   5,
   1,
   4,
   1,
   4,
   2,
   5,
   3,
   2,
   5,
   4,
   5],
  ['2025-07-18 05:48:48',
   2376,
   '2a',
   2,
   3,
   4,
   3.0,
   4,
   3,
   3,
   2,
   2,
   3,
   5,
   5,
   1,
   2,
   5,
   4,
   1,
   1,
   5,
   1,
   4,
   1,
   3,
   3,
   5,
   3,
   2,
   5,
   4,
   3],
  ['2025-07-17 18:45:15',
   2584,
   '2a',
   2,
   3,
   4,
   3.0,
   4,
   1,
   3,
   2,
   2,
   3,
   1,
   5,
   2,
   3,
   5,
   4,
   5,
   1,
   1,
   1,
   4,
   1,
   5,
   2,
   5,
   3,
   2,
   5,
   4,
   3],
  ['2025-07-17 21:49:04',
   2168,
   '2a',
   2,
   3,
   4,
   3.0,
   4,
   1,
   3,
   2,
   1,
   3,
   5,
   5,
   1,
   2,
   5,
   3,
   1,
   3,
   5,
   1,
   4,
   2,
   3,
   2,
   5,
   3,
   2,
   5,
   4,
   3],
  ['2025-07-17 22:27:44',
   2272,
   '2a',
   2,
   3,
   4,
   3.0,
   4,
   1,
   3,
 

In [104]:
sequences = df.iloc[:,1:].values.tolist()

student_clusters = {}

for cluster_id, seqs in clusters.items():

    for seq in seqs:

        idx = sequences.index(seq)

        student_clusters[df.iloc[idx]["estudante"]] = cluster_id

student_clusters

{np.int64(1532): 0,
 np.int64(194): 0,
 np.int64(1302): 0,
 np.int64(561): 0,
 np.int64(1668): 0,
 np.int64(590): 0,
 np.int64(1575): 0,
 np.int64(1714): 0,
 np.int64(360): 0,
 np.int64(1270): 0,
 np.int64(1415): 0,
 np.int64(1624): 0,
 np.int64(1551): 0,
 np.int64(1196): 0,
 np.int64(1050): 0,
 np.int64(955): 0,
 np.int64(1246): 0,
 np.int64(132): 0,
 np.int64(1242): 0,
 np.int64(1481): 0,
 np.int64(300): 0,
 np.int64(1377): 0,
 np.int64(1735): 0,
 np.int64(725): 0,
 np.int64(1334): 0,
 np.int64(213): 0,
 np.int64(886): 0,
 np.int64(72): 0,
 np.int64(702): 0,
 np.int64(1372): 0,
 np.int64(1675): 0,
 np.int64(1619): 0,
 np.int64(493): 0,
 np.int64(1682): 0,
 np.int64(139): 0,
 np.int64(1411): 0,
 np.int64(1330): 0,
 np.int64(688): 0,
 np.int64(141): 0,
 np.int64(1090): 0,
 np.int64(220): 0,
 np.int64(943): 0,
 np.int64(1620): 0,
 np.int64(1017): 0,
 np.int64(1684): 0,
 np.int64(1518): 0,
 np.int64(1641): 0,
 np.int64(692): 0,
 np.int64(1468): 0,
 np.int64(1042): 0,
 np.int64(1305): 0,


In [105]:
similarity_threshold = 90

suspects = []

for cluster_id in set(student_clusters.values()):

    students = [s for s,c in student_clusters.items() if c == cluster_id]

    for s1, s2 in combinations(students,2):

        seq1 = df[df.estudante==s1].iloc[:,1:].values.tolist()[0]
        seq2 = df[df.estudante==s2].iloc[:,1:].values.tolist()[0]

        sim = similarity(seq1,seq2)

        if sim >= similarity_threshold:

            suspects.append({
                "Student1":s1,
                "Student2":s2,
                "Similarity":sim
            })

In [106]:
report = pd.DataFrame(suspects)

report

,Student1,Student2,Similarity
0,1668,360,90.909091
1,1624,132,90.909091
2,1624,1242,90.909091
3,1624,1465,90.909091
4,955,1242,90.909091
5,702,1675,90.909091
6,1372,1518,90.909091
7,1675,141,90.909091
8,1293,1465,90.909091
9,463,1545,93.939394
